# Rete Pix2Pix
In questo notebook analizziamo il funzionamento dello script `Pix2Pix.py`, responsabile dell'addestramento e della validazione di una rete neurale generativa per la colorazione virtuale di immagini istopatologiche. Il codice implementa un'architettura Pix2Pix con generatore **U-Net** e discriminatore **PatchGAN**, addestrata su coppie di immagini allineate label-free \- stained.

## Passaggi preliminari

### Pacchetti necessari
I pacchetti necessari per lo script `Pix2Pix.py` sono:

- `numpy`
- `pytorch`
- `torchvision`

In [ ]:
import os, random, time, datetime, sys  # Pacchetti standard
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.utils import save_image

## Dataset
Per addestrare correttamente una rete Pix2Pix, è fondamentale disporre di dati organizzati in coppie di immagini perfettamente allineate in cui ad ogni immagine label-free (non colorata) corrisponde la sua controparte stained (colorata).
Nel nostro caso, i dati istopatologici sono organizzati in file `.tif`, dove ogni coppia condivide un prefisso comune nel nome (es. 00240_00100_label_free.tif e 00240_00100_stained.tif).

In questa sezione definiamo una classe `PairedHistologyDataset`, ereditaria della classe `Dataset` di **pytorch**, che identifica e carica automaticamente tutte le coppie di immagini presenti nella cartella fornita.

Questa struttura consente di gestire dataset di immagini istologiche accoppiate in modo efficiente e modulare, rendendole pronte per il training supervisionato della rete neurale.

In [ ]:
class PairedHistologyDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.pairs = self._get_pairs()

    def _get_pairs(self):
        files = os.listdir(self.folder_path)
        prefixes = [f.replace('_label_free.tif', '')
                    for f in files if f.endswith('_label_free.tif')
                    and f.replace('_label_free.tif', '') + '_stained.tif' in files]
        return sorted(prefixes)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prefix = self.pairs[idx]
        lf = Image.open(os.path.join(self.folder_path, prefix + '_label_free.tif')).convert('RGB')
        st = Image.open(os.path.join(self.folder_path, prefix + '_stained.tif')).convert('RGB')
        if self.transform:
            lf = self.transform(lf)
            st = self.transform(st)
        return lf, st

Le funzioni `__len__(self)` e `__getitem__(self, idx)` sono implementate per rispettare le specifiche della classe `Dataset` di **pytorch**. La funzione `__len__(self)` restituisce il numero di coppie di immagini nel dataset, mentre `__getitem__(self, idx)` carica e restituisce la coppia di immagini corrispondente all'indice `idx`.

## Generatore
Il generatore è il componente della rete che si occupa di tradurre un'immagine non colorata (label-free) in una versione virtualmente colorata, simulando l'effetto di una colorazione istopatologica H&E.
Per farlo, utilizziamo un'architettura chiamata U-Net, particolarmente efficace nelle trasformazioni image to image perché riesce a preservare sia le strutture locali che la coerenza globale.

In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self, n_channels=3, n_classes=3, bilinear=False):
        super(UNetGenerator, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        self.up1 = Up(1024, 512, bilinear)
        self.up2 = Up(512, 256, bilinear)
        self.up3 = Up(256, 128, bilinear)
        self.up4 = Up(128, 64, bilinear)

        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        logits = self.outc(x)
        return logits


La U-Net è composta da due parti principali:

- Encoder: comprime progressivamente l’immagine riducendo la sua risoluzione ma aumentando il numero di canali (profondità). In pratica, estrae caratteristiche astratte e informazioni ad alto livello.

- Decoder: ricostruisce l’immagine riportandola alla risoluzione originale, tentando di riprodurre una versione colorata realistica dell’input.

Ogni passaggio è realizzato tramite blocchi convoluzionali (filtri che estraggono pattern) e funzioni di attivazione (come ReLU), che trasformano i dati mantenendone la struttura.

Durante la compressione, però, si rischia di perdere dettagli importanti (es. bordi cellulari o nuclei).
Per questo, la U-Net utilizza delle connessioni di salto: collegamenti diretti tra ogni livello dell’encoder e il corrispondente livello del decoder.

Queste connessioni servono per riutilizzare i dettagli catturati all'inizio e "fonderli" nella fase di ricostruzione, migliorando la qualità dell'immagine generata.

Tecnicamente, ciò avviene concatenando le feature map in profondità lungo l’asse dei canali.

Durante la compressione, però, si rischia di perdere dettagli importanti (es. bordi cellulari o nuclei).
Per questo, la U-Net utilizza delle connessioni di salto: collegamenti diretti tra ogni livello dell’encoder e il corrispondente livello del decoder.

Queste connessioni servono per riutilizzare i dettagli catturati all'inizio e "fonderli" nella fase di ricostruzione, migliorando la qualità dell'immagine generata.

Tecnicamente, ciò avviene concatenando le feature map in profondità lungo l’asse dei canali.

## Discriminatore

## Funzioni di perdita

## Addestramento

### Validazione

### Checkpoint

## Test

### Metriche (loss)

## Risultati